# Lesson 5.3 — Classifying Singularities: Shoulder, Elbow, Wrist
**Module 6 · Unit 5 · Lesson 19**

A spherical wrist decouples J (det J = det J₁₁·det J₂₂). We build a 6-DOF arm, validate its Jacobian by finite differences, and show a **wrist** singularity (rank 6→5 at θ₅=0). Plus the **shoulder** mechanism and the **elbow** (extension) case.

In [1]:
import numpy as np
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i]); M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def Jv_planar(P,T,q): return geometric_jacobian(P,T,q)[:2,:]
def reach(P,T,q):
    M=forward_chain(P,T,q)[-1]; return float(np.hypot(M[0,3],M[1,3]))
P2=[(0,0,1.0,0),(0,0,1.0,0)]; T2=["R","R"]            # planar 2R, L1=L2=1


## A 6-DOF (Puma-like) arm, Jacobian validated by finite differences

In [2]:
checks=[]
def vee(S): return np.array([S[2,1],S[0,2],S[1,0]])
def fd_jac(P,T,q,eps=1e-6):
    n=len(q); J=np.zeros((6,n))
    for i in range(n):
        e=np.zeros(n); e[i]=eps
        Tp=forward_chain(P,T,q+e)[-1]; Tm=forward_chain(P,T,q-e)[-1]
        J[:3,i]=(Tp[:3,3]-Tm[:3,3])/(2*eps); M=Tp[:3,:3]@Tm[:3,:3].T; J[3:,i]=vee((M-M.T)/2)/(2*eps)
    return J
P6=[(0,0,0,np.pi/2),(0,0,1.0,0),(0,0,0,np.pi/2),(0,1.0,0,-np.pi/2),(0,0,0,np.pi/2),(0,0.3,0,0)]; T6=["R"]*6
q=np.array([0.3,0.5,-0.4,0.6,0.7,0.2])
checks.append(np.allclose(geometric_jacobian(P6,T6,q),fd_jac(P6,T6,q),atol=1e-5))
print("6R geometric J == finite-difference:",checks[-1])

6R geometric J == finite-difference: True


## Wrist singularity: set θ₅ = 0 → rank drops 6 → 5, σ_min → 0

In [3]:
q_reg=q.copy(); q_wri=q.copy(); q_wri[4]=0.0
r_reg=np.linalg.matrix_rank(geometric_jacobian(P6,T6,q_reg),tol=1e-6)
r_wri=np.linalg.matrix_rank(geometric_jacobian(P6,T6,q_wri),tol=1e-6)
s_wri=np.linalg.svd(geometric_jacobian(P6,T6,q_wri),compute_uv=False)
print("rank (regular):",r_reg,"  rank (theta5=0):",r_wri,"  sigma_min:",round(s_wri[-1],6))
print("-> wrist singularity: an ORIENTATION direction is lost")
checks.append(r_reg==6 and r_wri==5)

rank (regular): 6   rank (theta5=0): 5   sigma_min: 0.0
-> wrist singularity: an ORIENTATION direction is lost


## Shoulder mechanism & elbow (extension) case

In [4]:
# shoulder: joint axis z parallel to (o_n - o) -> z x (o_n-o)=0 (no position contribution)
z=np.array([0,0,1.0]); lever=np.array([0,0,2.0])
print("shoulder: z x (o_n-o) =",np.cross(z,lever)," (joint adds no tool-position velocity)")
checks.append(np.allclose(np.cross(z,lever),0))
# elbow: planar 2R straight -> arm (position) singularity, rank 1
checks.append(np.linalg.matrix_rank(Jv_planar(P2,T2,np.array([0.4,0.0])),tol=1e-6)<2)
print("elbow (straight 2R): position rank drops to",np.linalg.matrix_rank(Jv_planar(P2,T2,np.array([0.4,0.0])),tol=1e-6))
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

shoulder: z x (o_n-o) = [0. 0. 0.]  (joint adds no tool-position velocity)
elbow (straight 2R): position rank drops to 1
All checks passed.
